# Compartment calling (generator) — `comp.impute.hdf`

**This notebook produces no figure panel** — it is the prerequisite generator for the A/B
compartment scores that Fig 4E (`fig4/09.diffcomp_majortype`) and, after quantile-normalization
in `fig4/02.decay_compartment`, Fig 5A/B boundary analysis (`fig5/03.mCoverCompboundary`) read.

Method: fit a compartment PCA model **once** on the merged 100 kb imputed contact map, then
project every major-type imputed cool onto that model to get its per-bin PC1 (compartment) score.
The *differential*-compartment test (Fig 4E `bin_stats.hdf`) is done by **dcHiC** (external R,
fed these pre-computed PCs) — see `fig4/05`; it is not reproduced here.

## 📥 Required input files
- `{ENTEX_ROOT}/merged_cool_impute/100K/merged.cool` · merged 100 kb imputed map (model fit)
- `{ENTEX_ROOT}/merged_cool_impute/100K/L1/{ct}.cool` · per-major-type 100 kb imputed maps (from scHiCluster imputation)
- `{REF_ROOT}/hg38/hg38.100kbin.CpG.txt` · per-bin CpG density (orients PC1 sign so A=high CpG)
- `hg38.main.chrom.sizes` (autosomes) · `{ENTEX_ROOT}/L1color.tsv` (major-type names)

In [1]:
import os, sys
ENTEX_ROOT = os.environ.get("ENTEX_ROOT", "/large_storage/zhoulab/zhoujt/project/ENTEx")
REF_ROOT   = os.environ.get("REF_ROOT", "/large_storage/zhoulab/ref")
BOOK_ROOT  = os.environ.get("BOOK_ROOT", f"{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas")
sys.path.insert(0, BOOK_ROOT)
import repro_guard
os.chdir(f"{ENTEX_ROOT}/analysis")

In [2]:
import cooler
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from concurrent.futures import ProcessPoolExecutor, as_completed

indir_impute = f"{ENTEX_ROOT}/merged_cool_impute/100K/"
outdir = f"{ENTEX_ROOT}/analysis/compartment/"
res = 100000
chrom_sizes = cooler.read_chromsizes(f"{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes", all_names=True).iloc[:22]
L1meta = pd.read_csv(f"{ENTEX_ROOT}/L1color.tsv", sep="\t", header=0, index_col=0)
cpg = pd.read_csv(f"{REF_ROOT}/hg38/hg38.100kbin.CpG.txt", header=0, index_col=3, sep="\t")
cpg["CpG_density"] = cpg["14_user_patt_count"] / (cpg["13_seq_len"] - cpg["11_num_N"])

## Fit the compartment PCA model on the merged 100 kb map
Per chromosome: keep well-covered bins → observed/expected `E = Q/decay` → correlation matrix
`C = corrcoef(log2 E)` → `PCA`. Of the top 2 PCs pick the one with stronger A/B saddle
segregation, and orient its sign by correlation with CpG density.

In [3]:
Qall, binall = [], []
cool = cooler.Cooler(f"{indir_impute}merged.cool")
for c in chrom_sizes.index:
    Q = cool.matrix(balance=False, sparse=True).fetch(c).toarray()
    Q = Q - np.diag(np.diag(Q))
    rowsum = Q.sum(axis=0)
    p50, p99 = np.percentile(rowsum[rowsum > 0], 50), np.percentile(rowsum[rowsum > 0], 99)
    binfilter = rowsum > (p50 * 2 - p99)   # drop low-coverage bins
    binall.append(binfilter)
    Qall.append(Q[binfilter][:, binfilter])

modelall = []
for k, chrom in enumerate(chrom_sizes.index):
    Q = Qall[k].copy()
    decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
    E = np.zeros(Q.shape)
    row, col = np.diag_indices(E.shape[0])
    E[row, col] = 1
    for i in range(1, E.shape[0]):
        E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
    E = E + E.T
    C = np.corrcoef(np.log2(E + 0.001))
    C = np.clip(C, np.percentile(C, 5), np.percentile(C, 95))
    pca = PCA(n_components=2, svd_solver="arpack")
    pc = pca.fit_transform(C)
    cpgtmp = cpg.loc[cpg["#1_usercol"] == chrom, "CpG_density"].values[binall[k]]
    r = []
    for i in range(2):
        labels, _ = pd.qcut(pc[:, i], 50, labels=False, retbins=True)
        sad = np.array([[E[np.ix_(labels == a, labels == b)].sum() for a in range(50)] for b in range(50)])
        cnt = np.array([[(labels == a).sum() * (labels == b).sum() for a in range(50)] for b in range(50)])
        sad = sad / cnt
        r.append((sad[:10, :10].sum() + sad[-10:, -10:].sum()) / (sad[:10, -10:].sum() + sad[-10:, :10].sum()))
    i = 0 if r[0] > r[1] else 1
    sign = 1 if pearsonr(cpgtmp, pc[:, i])[0] > 0 else -1
    modelall.append(sign * pca.components_[i])

In [4]:
def compute_comp(cool_path):
    """Project one 100 kb cool onto the fitted compartment model → genome-wide PC1 (filtered bins)."""
    pcall = []
    cool = cooler.Cooler(cool_path)
    for k, chrom in enumerate(chrom_sizes.index):
        Q = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray()
        Q = Q - np.diag(np.diag(Q))
        Q = Q[binall[k]][:, binall[k]]
        decay = np.array([np.mean(np.diag(Q, i)) for i in range(Q.shape[0])])
        E = np.zeros(Q.shape)
        row, col = np.diag_indices(E.shape[0])
        E[row, col] = 1
        for i in range(1, E.shape[0]):
            E[row[:-i], col[i:]] = (Q[row[:-i], col[i:]] + 1e-5) / (decay[i] + 1e-5)
        E = E + E.T
        C = np.corrcoef(np.log2(E + 0.001))
        C = np.clip(C, np.percentile(C, 5), np.percentile(C, 95))
        pcall.append((C - np.mean(C, axis=0)).dot(modelall[k]))
    return np.concatenate(pcall)

## Project every major-type imputed cool → `comp.impute.hdf`
Rows = all 100 kb autosomal bins (`raw_binfilter` marks the covered ones); columns = the 35
major types. This table is read by `fig4/05` (dcHiC differential) and quantile-normalized in
`fig4/02` into `comp.impute.qnorm.hdf` (read by `fig5/03`).

In [5]:
binfilter = np.concatenate(binall)
result = {}
with ProcessPoolExecutor(16) as executor:
    futures = {executor.submit(compute_comp, f"{indir_impute}L1/{ct}.cool"): ct
               for ct in L1meta.index if os.path.isfile(f"{indir_impute}L1/{ct}.cool")}
    for future in as_completed(futures):
        result[futures[future]] = future.result()

comp = cooler.util.binnify(chrom_sizes, res)
comp["chrom"] = comp["chrom"].astype(str)
comp["raw_binfilter"] = binfilter
for ct in result:
    comp[ct] = 0.0
    comp.loc[binfilter, ct] = result[ct]
comp.to_hdf(f"{outdir}L1/comp.impute.hdf", key="data")
comp.shape